# colab_2 — Qwen2.5-VL-7B A100 파인튜닝
- 환경: Google Colab A100 40GB
- 모델: Qwen/Qwen2.5-VL-7B-Instruct (NF4 4bit)
- 목표: 재활용품 이미지 기반 VQA 4지선다

## colab_1 대비 수정사항
1. 패키지 설치 안정화 (scikit-learn 추가)
2. MAX_NEW_TOKENS 1 → 3
3. Stratified Split 적용 (정답 비율 유지)
4. Validation TypeError 수정 (labels 분리 후 명시적 전달, val_loss 유지)
5. TTA hflip 제거 (텍스트/마크 뒤집힘 방지), 동점 시 원본 우선
6. Step 12 오답분석 max_new_tokens 1 → 3


# Step 1. 패키지 설치
- 코랩 기본 torch 건드리지 않음 (A100은 stable로 충분)
- 버전 고정으로 충돌 방지
- 설치 후 런타임 자동 재시작 (재시작 후 Step 2부터 실행)

In [ ]:
import subprocess, sys

# 1. 충돌나는 패키지 먼저 제거
subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torchvision", "torchaudio", "triton"
])

# 2. 버전 고정해서 재설치
pkgs = [
    "torchvision==0.20.1",   # torch 2.5.1 호환 버전으로 고정
    "transformers==4.50.0",
    "accelerate==0.34.2",
    "peft==0.13.2",
    "bitsandbytes==0.45.5",
    "qwen-vl-utils==0.0.8",
    "tokenizers==0.21.0",
    "triton==3.1.0",
    "datasets",
    "pillow",
    "pandas",
    "tqdm",
    "psutil",
    "scikit-learn",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", *pkgs
])

print("✅ 설치 완료!")
print("⚠️ 런타임 > 세션 다시 시작 을 눌러주세요.")
print("   재시작 후 Step 2부터 실행하세요.")

# Step 2. 버전 확인 (재시작 후 여기서부터 실행)

In [ ]:
import torch, transformers, peft, accelerate
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name()}")
    print(f"VRAM:         {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB")

# Step 3. 구글 드라이브 마운트 + 데이터 압축 해제

In [ ]:
from google.colab import drive
import zipfile, os

drive.mount("/content/drive")

# ── 경로 설정 ──────────────────────────────────────────
COLAB_BASE  = "/content/drive/MyDrive"
ZIP_PATH    = f"{COLAB_BASE}/2026-ssafy-ai-15-2.zip"
EXTRACT_DIR = "/content/2026-ssafy-ai-15-2"   # /content에 풀기 (드라이브 I/O보다 3~5배 빠름)
SAVE_DIR    = f"{COLAB_BASE}/checkpoints/qwen2_5_vl_7b_lora"  # 체크포인트는 드라이브에 저장

# ── 압축 해제 (이미 풀려있으면 스킵) ───────────────────
if not os.path.exists(EXTRACT_DIR):
    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")
    print("압축 해제 완료!")
else:
    print("이미 압축 해제됨, 스킵")

# ── 경로 검증 ──────────────────────────────────────────
DATA_DIR = "/content"
for f in ["train.csv", "test.csv"]:
    path = os.path.join(DATA_DIR, f)
    status = "✅" if os.path.isfile(path) else "❌ 없음"
    print(f"{f}: {status}")
for d in ["train", "test"]:
    path = os.path.join(DATA_DIR, d)
    status = "✅" if os.path.isdir(path) else "❌ 없음"
    print(f"{d}/: {status}")

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"\nDATA_DIR:  {DATA_DIR}")
print(f"SAVE_DIR:  {SAVE_DIR}")

# Step 4. 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random, gc, logging
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Sampler
from dataclasses import dataclass
import torch
from typing import Dict, List, Any, Optional
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = 300_000_000

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ==========================================================
#  CONFIG — A100 코랩 최적화
# ==========================================================
CONFIG = {
    "MODEL_ID":            "Qwen/Qwen2.5-VL-7B-Instruct",
    # Dynamic resolution (28px 패치 단위, 720x960 원본 기준 최적)
    "MIN_PIXELS":          256 * 28 * 28,   # 200,704
    "MAX_PIXELS":          980 * 28 * 28,   # 768,320
    "EPOCHS":              10,
    "GRAD_ACCUM":          4,
    "LR":                  8e-6,
    "WEIGHT_DECAY":        0.01,
    "WARMUP_RATIO":        0.1,
    "PATIENCE":            5,
    "MAX_TRAIN_SAMPLES":   None,
    "MAX_GRAD_NORM":       0.5,
    "MAX_NEW_TOKENS":      3,   # 1 → 3 (출력 안정성)
    "SAVE_DIR":            SAVE_DIR,
    "SEED":                42,
    "NAN_SKIP_THRESHOLD":  3,
    "MEMORY_LOG_INTERVAL": 100,
    # LoRA
    "LORA_R":              24,
    "LORA_ALPHA":          48,
    "LORA_DROPOUT":        0.08,
}

MODEL_ID          = CONFIG["MODEL_ID"]
EPOCHS            = CONFIG["EPOCHS"]
GRAD_ACCUM        = CONFIG["GRAD_ACCUM"]
LR                = CONFIG["LR"]
WEIGHT_DECAY      = CONFIG["WEIGHT_DECAY"]
WARMUP_RATIO      = CONFIG["WARMUP_RATIO"]
PATIENCE          = CONFIG["PATIENCE"]
MAX_TRAIN_SAMPLES = CONFIG["MAX_TRAIN_SAMPLES"]
MAX_NEW_TOKENS    = CONFIG["MAX_NEW_TOKENS"]
SAVE_DIR          = CONFIG["SAVE_DIR"]
SEED              = CONFIG["SEED"]
MAX_GRAD_NORM     = CONFIG["MAX_GRAD_NORM"]

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info(f"Seed set to {seed} (deterministic mode ON)")

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def log_gpu_memory(tag: str = ""):
    if torch.cuda.is_available():
        alloc    = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        logger.info(f"[GPU Memory - {tag}] Allocated: {alloc:.2f}GB | Reserved: {reserved:.2f}GB")

def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── 데이터 로드 + 검증 ─────────────────────────────────
REQUIRED_TRAIN_COLS = {"id", "question", "a", "b", "c", "d", "answer", "path"}
REQUIRED_TEST_COLS  = {"id", "question", "a", "b", "c", "d", "path"}

train_csv = os.path.join(DATA_DIR, "train.csv")
test_csv  = os.path.join(DATA_DIR, "test.csv")

for csv_path in [train_csv, test_csv]:
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"필수 데이터 파일 없음: {csv_path}")

train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

missing_train = REQUIRED_TRAIN_COLS - set(train_df.columns)
missing_test  = REQUIRED_TEST_COLS  - set(test_df.columns)
if missing_train:
    raise ValueError(f"train.csv 필수 컬럼 누락: {missing_train}")
if missing_test:
    raise ValueError(f"test.csv 필수 컬럼 누락: {missing_test}")

# ── 이미지 경로 절대경로로 변환 ────────────────────────
def resolve_path(p, base):
    p = str(p)
    return p if os.path.isabs(p) else os.path.join(base, p)

train_df["path"] = train_df["path"].apply(lambda p: resolve_path(p, DATA_DIR))
test_df["path"]  = test_df["path"].apply(lambda p: resolve_path(p, DATA_DIR))

# ── 이미지 경로 존재 검증 ──────────────────────────────
def validate_image_paths(df, name):
    missing_mask = ~df["path"].apply(os.path.isfile)
    n_missing = missing_mask.sum()
    if n_missing > 0:
        logger.warning(f"[{name}] {n_missing}개 이미지 파일 누락 → 해당 행 제거")
        for p in df.loc[missing_mask, "path"].head(5).tolist():
            logger.warning(f"  누락: {p}")
        df = df[~missing_mask].reset_index(drop=True)
    return df

train_df = validate_image_paths(train_df, "train")
test_df  = validate_image_paths(test_df,  "test")

# ── 정답 레이블 검증 ───────────────────────────────────
valid_answers = {"a", "b", "c", "d"}
invalid_mask  = ~train_df["answer"].str.strip().str.lower().isin(valid_answers)
if invalid_mask.sum() > 0:
    logger.warning(f"유효하지 않은 정답 {invalid_mask.sum()}개 → 제거")
    train_df = train_df[~invalid_mask].reset_index(drop=True)

logger.info(f"train.csv: {len(train_df)}행")
logger.info(f"test.csv:  {len(test_df)}행")

if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(n=MAX_TRAIN_SAMPLES, random_state=SEED)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"학습 데이터: {len(train_df):,}개 | 테스트 데이터: {len(test_df):,}개")
print("\n학습 정답 분포:")
print(train_df["answer"].str.strip().str.lower().value_counts().sort_index())

# Step 5. 모델 & Processor 로드

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Dynamic resolution 활성화 (min != max)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=CONFIG["MIN_PIXELS"],
    max_pixels=CONFIG["MAX_PIXELS"],
    trust_remote_code=True,
)

try:
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
except Exception as e:
    logger.error(f"모델 로드 실패: {e}")
    raise RuntimeError(
        f"모델 '{MODEL_ID}' 로드 실패. 네트워크/HF_TOKEN/GPU 메모리 확인"
    ) from e

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=CONFIG["LORA_R"],
    lora_alpha=CONFIG["LORA_ALPHA"],
    lora_dropout=CONFIG["LORA_DROPOUT"],
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
log_gpu_memory("모델 로드 완료")

# Step 6. 프롬프트 템플릿

In [ ]:
# 도메인 특화 시스템 프롬프트
SYSTEM_INSTRUCT = (
    "You are an expert in recycling and waste material classification. "
    "Analyze the image carefully, paying attention to material texture, "
    "shape, color, and surface details. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    def _safe(v):
        try:
            return str(v) if pd.notna(v) else ""
        except (TypeError, ValueError):
            return str(v) if v is not None else ""
    question = _safe(question)
    a, b, c, d = _safe(a), _safe(b), _safe(c), _safe(d)
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Step 7. Dataset & DataCollator

In [ ]:
import random
from torchvision import transforms as T

# 이미지 증강 — 재활용품 소재 색상/질감 보존을 위해 보수적으로
AUGMENT_TRANSFORM = T.Compose([
    T.RandomApply([T.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02
    )], p=0.5),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.2),
    T.RandomApply([T.RandomAffine(
        degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)
    )], p=0.3),
])

def augment_image(img: Image.Image) -> Image.Image:
    try:
        return AUGMENT_TRANSFORM(img)
    except Exception as e:
        logger.warning(f"이미지 증강 실패 (원본 사용): {e}")
        return img

CHOICE_KEYS = ["a", "b", "c", "d"]

def shuffle_choices(row: dict) -> dict:
    options  = {k: str(row[k]) for k in CHOICE_KEYS}
    gold_key = str(row["answer"]).strip().lower()
    if gold_key not in CHOICE_KEYS:
        logger.warning(f"shuffle_choices: 유효하지 않은 answer='{gold_key}', 셔플 스킵")
        return {k: options[k] for k in CHOICE_KEYS}
    shuffled_keys = CHOICE_KEYS.copy()
    random.shuffle(shuffled_keys)
    new_options  = {new_k: options[old_k] for new_k, old_k in zip(CHOICE_KEYS, shuffled_keys)}
    new_gold_idx = shuffled_keys.index(gold_key)
    new_gold     = CHOICE_KEYS[new_gold_idx]
    return {**new_options, "answer": new_gold}

def safe_load_image(path: str) -> Optional[Image.Image]:
    if not os.path.isfile(path):
        logger.warning(f"이미지 파일 없음: {path}")
        return None
    try:
        img = Image.open(path).convert("RGB")
        img.load()
        return img
    except Exception as e:
        logger.warning(f"이미지 로드 실패: {path} | {e}")
        return None


class VQAMCDataset(Dataset):
    def __init__(
        self, df, processor,
        train: bool = True,
        aug_image: bool = True,
        aug_choice: bool = True,
        aug_prob: float = 0.3,   # 0.5 → 0.3 (소재 색상 보존)
    ):
        self.df          = df.reset_index(drop=True)
        self.processor   = processor
        self.train       = train
        self.aug_image   = aug_image and train
        self.aug_choice  = aug_choice and train
        self.aug_prob    = aug_prob
        self._skip_count = 0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i].to_dict()
        img = safe_load_image(str(row.get("path", "")))
        if img is None:
            self._skip_count += 1
            logger.warning(f"샘플 {i} 스킵 (누적 {self._skip_count}개)")
            return self._make_dummy_item()

        if self.aug_image and random.random() < self.aug_prob:
            img = augment_image(img)
        if self.aug_choice and random.random() < self.aug_prob:
            row = {**row, **shuffle_choices(row)}

        q    = str(row.get("question", ""))
        a, b, c, d = str(row.get("a","")), str(row.get("b","")), str(row.get("c","")), str(row.get("d",""))
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]

        try:
            if self.train:
                return self._build_train_item(messages, img, row)
            else:
                return self._build_infer_item(messages, img)
        except Exception as e:
            logger.warning(f"샘플 {i} 처리 실패: {e}")
            self._skip_count += 1
            return self._make_dummy_item()

    def _build_train_item(self, messages, img, row):
        gold = str(row.get("answer", "a")).strip().lower()
        if gold not in CHOICE_KEYS:
            logger.warning(f"유효하지 않은 answer='{gold}', 'a'로 대체")
            gold = "a"

        full_messages = messages + [
            {"role": "assistant", "content": [{"type": "text", "text": gold}]}
        ]
        full_text = self.processor.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        )
        full_enc = self.processor(
            text=[full_text], images=[img], return_tensors="pt", padding=False
        )
        item = {k: v.squeeze(0) for k, v in full_enc.items()}

        prompt_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt_enc = self.processor(
            text=[prompt_text], images=[img], return_tensors="pt", padding=False
        )
        prompt_len = prompt_enc["input_ids"].shape[1]
        total_len  = item["input_ids"].shape[0]

        if prompt_len >= total_len:
            logger.warning(f"prompt_len({prompt_len}) >= total_len({total_len}) → 더미 반환")
            del img
            return self._make_dummy_item()

        labels = item["input_ids"].clone()
        labels[:prompt_len] = -100

        n_valid = int((labels != -100).sum().item())
        if n_valid == 0:
            logger.warning(f"labels 전부 -100 → 더미 반환")
            del img
            return self._make_dummy_item()

        item["labels"] = labels
        del img
        return item

    def _build_infer_item(self, messages, img):
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        enc = self.processor(
            text=[text], images=[img], return_tensors="pt", padding=False
        )
        del img
        return {k: v.squeeze(0) for k, v in enc.items()}

    def _make_dummy_item(self):
        pad_id = self.processor.tokenizer.pad_token_id or 0
        dummy_ids  = torch.tensor([pad_id], dtype=torch.long)
        dummy_mask = torch.tensor([0],      dtype=torch.long)
        item = {"input_ids": dummy_ids, "attention_mask": dummy_mask}
        if self.train:
            item["labels"] = torch.tensor([-100], dtype=torch.long)
        return item


# ── LengthGroupedSampler (패딩 낭비 최소화 → 속도 개선) ────
class LengthGroupedSampler(Sampler):
    """파일 크기 기준으로 비슷한 길이의 샘플끼리 배치 구성"""
    def __init__(self, dataset, batch_size, seed):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.seed       = seed

    def __iter__(self):
        indices = sorted(
            range(len(self.dataset)),
            key=lambda i: os.path.getsize(self.dataset.df.iloc[i]["path"])
        )
        batches = [
            indices[i:i + self.batch_size]
            for i in range(0, len(indices), self.batch_size)
        ]
        random.shuffle(batches)
        return iter([idx for batch in batches for idx in batch])

    def __len__(self):
        return len(self.dataset)


# ── DataCollator ────────────────────────────────────────────
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        if not features:
            raise ValueError("DataCollator에 빈 features 리스트 전달됨")

        has_labels  = self.train and "labels" in features[0]
        labels_list = None
        if has_labels:
            labels_list = [f.pop("labels") for f in features]

        if len(features) == 1:
            batch = {k: v.unsqueeze(0) for k, v in features[0].items()}
            if has_labels:
                batch["labels"] = labels_list[0].unsqueeze(0)
            return batch

        input_ids_list = [f["input_ids"]     for f in features]
        attn_list      = [f["attention_mask"] for f in features]
        max_len = max(ids.shape[0] for ids in input_ids_list)

        pad_id = self.processor.tokenizer.pad_token_id or 0
        padded_ids  = torch.full((len(features), max_len), pad_id, dtype=torch.long)
        padded_mask = torch.zeros(len(features), max_len,           dtype=torch.long)
        for i, (ids, mask) in enumerate(zip(input_ids_list, attn_list)):
            padded_ids[i,  :ids.shape[0]]  = ids
            padded_mask[i, :mask.shape[0]] = mask

        batch = {"input_ids": padded_ids, "attention_mask": padded_mask}

        if "pixel_values" in features[0]:
            batch["pixel_values"] = torch.cat([f["pixel_values"] for f in features], dim=0)
        if "image_grid_thw" in features[0]:
            grids = [f["image_grid_thw"] for f in features]
            batch["image_grid_thw"] = (
                torch.stack(grids, dim=0) if grids[0].dim() == 1
                else torch.cat(grids, dim=0)
            )

        if has_labels:
            pad_lab = torch.full((len(features), max_len), -100, dtype=torch.long)
            for i, lab in enumerate(labels_list):
                pad_lab[i, :lab.shape[0]] = lab
            batch["labels"] = pad_lab

        return batch

# Step 8. DataLoader

In [ ]:
from sklearn.model_selection import train_test_split

def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

g = torch.Generator()
g.manual_seed(SEED)

# Stratified Split — 정답 비율 유지 (수정 3)
train_subset, valid_subset = train_test_split(
    train_df,
    test_size=0.05,
    random_state=SEED,
    stratify=train_df["answer"]
)
train_subset = train_subset.reset_index(drop=True)
valid_subset = valid_subset.reset_index(drop=True)

train_ds = VQAMCDataset(
    train_subset, processor,
    train=True, aug_image=True, aug_choice=True, aug_prob=0.3
)
valid_ds = VQAMCDataset(
    valid_subset, processor,
    train=False,
    aug_image=False, aug_choice=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=2,
    sampler=LengthGroupedSampler(train_ds, batch_size=2, seed=SEED),
    collate_fn=DataCollator(processor, train=True),
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=2,
    shuffle=False,
    collate_fn=DataCollator(processor, train=True),  # labels 필요
    num_workers=2,
    pin_memory=True,
)

print(f"Train: {len(train_ds):,}개 | Valid: {len(valid_ds):,}개")
print(f"Train batches: {len(train_loader):,} | Valid batches: {len(valid_loader):,}")


# Step 9. Fine-tuning

In [ ]:
import math, shutil

model = model.to(device)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    logger.warning("bf16 미지원 GPU — fp16 폴백. loss underflow 위험 있음")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch    = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * steps_per_epoch

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * WARMUP_RATIO),
    num_training_steps=num_training_steps,
)

def save_checkpoint_safe(model, processor, save_dir, metadata=None):
    tmp_dir    = save_dir + "_tmp"
    backup_dir = save_dir + "_backup"
    try:
        os.makedirs(tmp_dir, exist_ok=True)
        model.save_pretrained(tmp_dir)
        processor.save_pretrained(tmp_dir)
        if metadata:
            import json as _json
            with open(os.path.join(tmp_dir, "training_metadata.json"), "w", encoding="utf-8") as mf:
                _json.dump(metadata, mf, indent=2, ensure_ascii=False, default=str)
        if os.path.exists(save_dir):
            if os.path.exists(backup_dir):
                shutil.rmtree(backup_dir)
            os.rename(save_dir, backup_dir)
        os.rename(tmp_dir, save_dir)
        if os.path.exists(backup_dir):
            shutil.rmtree(backup_dir)
        logger.info(f"체크포인트 저장 완료: {save_dir}")
    except Exception as e:
        logger.error(f"체크포인트 저장 실패: {e}")
        if os.path.exists(tmp_dir):
            shutil.rmtree(tmp_dir, ignore_errors=True)

log_gpu_memory("학습 시작 전")

best_val_acc     = 0.0
patience_counter = 0
global_step      = 0
nan_counter      = 0
val_acc_history  = []   # moving average용

for epoch in range(EPOCHS):

    # ── Train ──────────────────────────────────────────────
    model.train()
    running_loss    = 0.0
    epoch_loss_sum  = 0.0
    epoch_loss_count = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        if step <= 10 and "labels" in batch:
            n_valid = int((batch["labels"] != -100).sum().item())
            if n_valid == 0:
                logger.error(f"[Step {step}] labels 전부 -100! label masking 오류")
            else:
                logger.info(f"[Step {step}] 유효 label 토큰 수: {n_valid} ✓")

        try:
            with torch.amp.autocast(
                "cuda",
                enabled=(device.type == "cuda"),
                dtype=torch.bfloat16 if use_bf16 else torch.float16
            ):
                outputs  = model(**batch)
                raw_loss = outputs.loss

                if torch.isnan(raw_loss) or torch.isinf(raw_loss):
                    nan_counter += 1
                    logger.warning(f"Step {step}: loss NaN/Inf, 스킵 (연속 {nan_counter}회)")
                    optimizer.zero_grad(set_to_none=True)
                    if nan_counter >= CONFIG["NAN_SKIP_THRESHOLD"]:
                        logger.error(f"NaN 연속 {nan_counter}회, 학습 중단")
                        break
                    continue
                nan_counter = 0
                loss = raw_loss / GRAD_ACCUM

            loss.backward()
            running_loss     += raw_loss.item()
            epoch_loss_sum   += raw_loss.item()
            epoch_loss_count += 1

        except RuntimeError as oom_err:
            if "out of memory" in str(oom_err).lower():
                logger.warning(f"OOM at step {step}, 캐시 정리 후 스킵")
                optimizer.zero_grad(set_to_none=True)
                del batch
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise

        if step % GRAD_ACCUM == 0:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            gn_val = grad_norm.item() if hasattr(grad_norm, "item") else float(grad_norm)
            pbar.set_postfix({
                "loss":  f"{running_loss / GRAD_ACCUM:.4f}",
                "lr":    f"{scheduler.get_last_lr()[0]:.2e}",
                "gnorm": f"{gn_val:.2f}",
            })
            running_loss = 0.0

            if global_step % 50 == 0:
                torch.cuda.empty_cache()
            if global_step % CONFIG["MEMORY_LOG_INTERVAL"] == 0:
                log_gpu_memory(f"step {global_step}")

    else:
        if step % GRAD_ACCUM != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    if nan_counter >= CONFIG["NAN_SKIP_THRESHOLD"]:
        logger.error("NaN 연속 발생으로 학습 중단")
        break

    avg_train_loss = epoch_loss_sum / max(epoch_loss_count, 1)
    logger.info(f"Epoch {epoch+1} avg train loss: {avg_train_loss:.4f}")

    # ── Validation ─────────────────────────────────────────
    model.eval()
    val_loss   = 0.0
    val_steps  = 0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [valid]"):
            vb = {k: v.to(device) for k, v in vb.items()}

            # labels 분리 후 명시적 전달 → val_loss 정상 계산 (수정 4)
            labels = vb.pop("labels")
            with torch.amp.autocast(
                "cuda",
                enabled=(device.type == "cuda"),
                dtype=torch.bfloat16 if use_bf16 else torch.float16
            ):
                out = model(**vb, labels=labels)

            if not (torch.isnan(out.loss) or torch.isinf(out.loss)):
                val_loss  += out.loss.item()
                val_steps += 1

            logits = out.logits

            for b_idx in range(labels.size(0)):
                valid_pos = (labels[b_idx] != -100).nonzero(as_tuple=True)[0]
                if len(valid_pos) == 0:
                    continue
                ans_pos  = valid_pos[0].item()
                pred_pos = ans_pos - 1
                if pred_pos < 0 or pred_pos >= logits.size(1):
                    continue
                pred_token = processor.tokenizer.decode(
                    [logits[b_idx, pred_pos].argmax().item()]
                ).strip().lower()
                true_token = processor.tokenizer.decode(
                    [labels[b_idx, ans_pos].item()]
                ).strip().lower()
                if true_token in ["a", "b", "c", "d"]:
                    correct += int(pred_token == true_token)
                    total   += 1

            del out, logits

    clear_gpu_cache()
    torch.cuda.empty_cache()

    avg_val_loss = val_loss / max(val_steps, 1)
    val_acc      = correct / max(total, 1)

    # ── Moving average (3 epoch) ────────────────────────────
    val_acc_history.append(val_acc)
    smoothed_acc = sum(val_acc_history[-3:]) / len(val_acc_history[-3:])

    # ── 에폭별 결과 출력 ────────────────────────────────────
    print(
        f"[Epoch {epoch+1}/{EPOCHS}] "
        f"train_loss={avg_train_loss:.4f} "
        f"val_loss={avg_val_loss:.4f} "
        f"val_acc={val_acc:.4f} ({correct}/{total}) "
        f"smoothed_acc={smoothed_acc:.4f}"
    )

    # ── Best Model 저장 (smoothed_acc 기준) ─────────────────
    if smoothed_acc > best_val_acc:
        print(f"  ✅ Smoothed Acc improved {best_val_acc:.4f} → {smoothed_acc:.4f}")
        best_val_acc     = smoothed_acc
        patience_counter = 0
        save_checkpoint_safe(model, processor, SAVE_DIR, metadata={
            "epoch":        epoch + 1,
            "val_acc":      val_acc,
            "smoothed_acc": smoothed_acc,
            "val_loss":     avg_val_loss,
            "global_step":  global_step,
            "config":       CONFIG,
        })
    else:
        patience_counter += 1
        print(f"  No improvement | patience {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("  Early Stopping")
            break

print(f"\nTraining Done | Best smoothed_acc: {best_val_acc:.4f}")
log_gpu_memory("학습 완료")

# Step 10. Inference (TTA 적용)

In [ ]:
from collections import Counter
from torchvision.transforms import functional as TF

def extract_choice(text: str) -> str:
    if not text or not isinstance(text, str):
        return random.choice(["a", "b", "c", "d"])
    text  = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return random.choice(["a", "b", "c", "d"])
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    for tok in last.split():
        if tok in ["a", "b", "c", "d"]:
            return tok
    for ch in reversed(text.replace("\n", " ").split()):
        if ch in ["a", "b", "c", "d"]:
            return ch
    return random.choice(["a", "b", "c", "d"])


def infer_single(img, messages):
    """이미지 1장 단일 추론"""
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    return extract_choice(processor.batch_decode(out_ids, skip_special_tokens=True)[0])


def infer_with_tta(img, row):
    """TTA: 원본 + 좌우반전 + 밝기 조정 → 다수결"""
    def make_messages(image):
        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        return [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": user_text},
            ]},
        ]

    votes = []
    # 원본
    votes.append(infer_single(img, make_messages(img)))
    # 밝기 조정 (hflip 제거 — 텍스트/마크 뒤집힘 방지) (수정 5)
    img_bright = TF.adjust_brightness(img, 1.2)
    votes.append(infer_single(img_bright, make_messages(img_bright)))

    # 동점 시 원본 결과 우선
    counter = Counter(votes)
    if counter.most_common(1)[0][1] == 1:
        return votes[0]
    return counter.most_common(1)[0][0]


# ── Best Model 로드 ─────────────────────────────────────
if os.path.exists(SAVE_DIR):
    logger.info(f"Best model adapter 로드: {SAVE_DIR}")
    try:
        model.load_adapter(SAVE_DIR, adapter_name="default")
        logger.info("Best model adapter 로드 완료")
    except Exception as e:
        logger.warning(f"Adapter 로드 실패, 현재 모델로 추론: {e}")
else:
    logger.warning(f"저장된 모델 없음 ({SAVE_DIR}). 현재 모델로 추론")

set_seed(SEED)
model.eval()
preds       = []
skipped_ids = []
clear_gpu_cache()

for i in tqdm(range(len(test_df)), desc="Inference (TTA)", unit="sample"):
    row = test_df.iloc[i]
    img = safe_load_image(str(row["path"]))

    if img is None:
        logger.warning(f"추론 샘플 {i} 이미지 로드 실패, 랜덤 예측")
        preds.append(random.choice(["a", "b", "c", "d"]))
        skipped_ids.append(row.get("id", i))
        continue

    try:
        pred = infer_with_tta(img, row)
        preds.append(pred)
    except Exception as e:
        logger.warning(f"추론 샘플 {i} 처리 실패: {e}, 랜덤 예측")
        preds.append(random.choice(["a", "b", "c", "d"]))
        skipped_ids.append(row.get("id", i))

    del img
    if (i + 1) % 100 == 0:
        clear_gpu_cache()

# ── 제출 파일 생성 ──────────────────────────────────────
if len(preds) != len(test_df):
    raise ValueError(f"예측 수({len(preds)}) ≠ 테스트 수({len(test_df)})")

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission_path = os.path.join(COLAB_BASE, "submission.csv")
submission.to_csv(submission_path, index=False)
print(f"submission.csv 저장 완료: {submission_path}")

if skipped_ids:
    print(f"이미지 로드 실패로 랜덤 예측: {len(skipped_ids)}개")

dist = submission["answer"].value_counts().sort_index()
print("\n예측 분포:")
print(dist)
pct = dist.values / dist.values.sum() * 100
print(f"분포 균형도 (이상적=25.0%): {pct.round(1)}")

# Step 11. 메모리 프로파일링

In [ ]:
def memory_profiling_report():
    import psutil
    print("=" * 60)
    print("  메모리 프로파일링 결과")
    print("=" * 60)
    ram = psutil.virtual_memory()
    print(f"\n[RAM]")
    print(f"  총 메모리:  {ram.total / 1e9:.1f} GB")
    print(f"  사용 중:    {ram.used / 1e9:.1f} GB ({ram.percent}%)")
    print(f"  가용:       {ram.available / 1e9:.1f} GB")
    process  = __import__("psutil").Process(os.getpid())
    proc_mem = process.memory_info()
    print(f"\n[Python Process]")
    print(f"  RSS: {proc_mem.rss / 1e9:.2f} GB | VMS: {proc_mem.vms / 1e9:.2f} GB")
    if torch.cuda.is_available():
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        allocated  = torch.cuda.memory_allocated() / 1e9
        reserved   = torch.cuda.memory_reserved() / 1e9
        peak       = torch.cuda.max_memory_allocated() / 1e9
        print(f"\n[GPU: {torch.cuda.get_device_name()}]")
        print(f"  총 VRAM:  {total_vram:.1f} GB")
        print(f"  할당됨:   {allocated:.2f} GB ({allocated/total_vram*100:.1f}%)")
        print(f"  예약됨:   {reserved:.2f} GB ({reserved/total_vram*100:.1f}%)")
        print(f"  Peak:     {peak:.2f} GB ({peak/total_vram*100:.1f}%)")
        stats = torch.cuda.memory_stats()
        if "num_ooms" in stats:
            print(f"  OOM 횟수: {stats['num_ooms']}")
    print("\n" + "=" * 60)
    print("  안정성 지표")
    print("=" * 60)
    print(f"  SEED:           {CONFIG['SEED']}")
    print(f"  GRAD_ACCUM:     {CONFIG['GRAD_ACCUM']}")
    print(f"  MAX_GRAD_NORM:  {CONFIG['MAX_GRAD_NORM']}")
    print(f"  LORA_R:         {CONFIG['LORA_R']}")
    print(f"  MIN_PIXELS:     {CONFIG['MIN_PIXELS']}")
    print(f"  MAX_PIXELS:     {CONFIG['MAX_PIXELS']}")
    print("=" * 60)

try:
    memory_profiling_report()
except ImportError:
    print("psutil 미설치")
    if torch.cuda.is_available():
        print(f"GPU Allocated: {torch.cuda.memory_allocated()/1e9:.2f}GB")
        print(f"GPU Reserved:  {torch.cuda.memory_reserved()/1e9:.2f}GB")
        print(f"GPU Peak:      {torch.cuda.max_memory_allocated()/1e9:.2f}GB")

# Step 12. 오답 분석

In [ ]:
model.eval()
results = []

with torch.no_grad():
    for idx in tqdm(range(len(valid_subset)), desc="오답 분석 중"):
        row = valid_subset.iloc[idx]
        img = safe_load_image(str(row["path"]))
        if img is None:
            continue

        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        messages  = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]
        text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        with torch.amp.autocast("cuda", enabled=True, dtype=torch.bfloat16):
            out_ids = model.generate(**inputs, max_new_tokens=3, do_sample=False)

        pred = extract_choice(processor.batch_decode(out_ids, skip_special_tokens=True)[0])
        gold = str(row["answer"]).strip().lower()

        results.append({
            "idx":      idx,
            "question": row["question"][:80],
            "gold":     gold,
            "pred":     pred,
            "correct":  pred == gold,
            "path":     row["path"],
        })
        del img
        if (idx + 1) % 100 == 0:
            clear_gpu_cache()

df_results = pd.DataFrame(results)
wrong      = df_results[~df_results["correct"]]

print(f"\n=== 전체: {len(df_results)} | 정답: {df_results['correct'].sum()} | 오답: {len(wrong)} ===")
print(f"정확도: {df_results['correct'].mean():.4f}")

print("\n=== Confusion Matrix (행=정답, 열=예측) ===")
print(pd.crosstab(df_results["gold"], df_results["pred"], margins=True))

print("\n=== 선택지별 정확도 ===")
for key in ["a", "b", "c", "d"]:
    sub = df_results[df_results["gold"] == key]
    if len(sub) > 0:
        print(f"  {key}: {sub['correct'].mean():.3f} ({sub['correct'].sum()}/{len(sub)})")

print(f"\n=== 오답 샘플 (상위 20개) ===")
for _, r in wrong.head(20).iterrows():
    print(f"  [{r['idx']}] 정답={r['gold']} 예측={r['pred']} | {r['question']}")